In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [2]:
df = pd.read_csv('dataset.csv')

In [3]:
DROP_COLS = [
    'comment', 'map_file', 'meta_data', 'direction', 'route_path',
    'citizen_accident_id', 'assigned_to_police_id',
    'age_of_truck', 'reason_breakdown', 'cargo_material',
    'resolved_at_address', 'resolved_at_latitude', 'resolved_at_longitude',
    'resolved_by_id', 'resolved_datetime',
    'description', 'kgid', 'veh_no',
    'id', 'address', 'end_address',  # free text / ID
    'map_file', 'last_modified_by_id', 'created_by_id',
    'modified_datetime', 'created_date',  # leakage risk / not features
    'closed_by_id', 'police_station', 'client_id', 'authenticated'
]
df_clean = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

In [4]:
df_clean['closed_datetime']   = pd.to_datetime(df_clean['closed_datetime'],   utc=True, errors='coerce')
df_clean['start_datetime']    = pd.to_datetime(df_clean['start_datetime'],     utc=True, errors='coerce')
df_clean['resolved_datetime_fallback'] = pd.to_datetime(
    df.get('resolved_datetime', pd.Series()), utc=True, errors='coerce'
)

# Use closed_datetime first, fall back to resolved_datetime
df_clean['end_ts'] = df_clean['closed_datetime'].fillna(df_clean['resolved_datetime_fallback'])
df_clean['duration_mins'] = (
    df_clean['end_ts'] - df_clean['start_datetime']
).dt.total_seconds() / 60

In [5]:
df_clean['severity'] = pd.cut(
    df_clean['duration_mins'],
    bins=[0, 30, 90, 240, 1440],
    labels=['low', 'medium', 'high', 'critical']
)

In [6]:
df_clean['hour']        = df_clean['start_datetime'].dt.hour
df_clean['day_of_week'] = df_clean['start_datetime'].dt.dayofweek
df_clean['month']       = df_clean['start_datetime'].dt.month
df_clean['is_weekend']  = df_clean['day_of_week'].isin([5, 6]).astype(int)
df_clean['is_peak']     = df_clean['hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)
df_clean['is_night']    = df_clean['hour'].isin(range(22, 24)).astype(int)

# ── 5. Geo features ───────────────────────────────────────────────────────────
df_clean['lat_bin'] = pd.cut(df_clean['latitude'],  bins=8, labels=False)
df_clean['lon_bin'] = pd.cut(df_clean['longitude'], bins=8, labels=False)
df_clean['has_end_location'] = (df_clean['endlatitude'].fillna(0) != 0).astype(int)


In [7]:
CAT_FILL_UNKNOWN = ['zone', 'junction', 'gba_identifier', 'veh_type', 'corridor']
for col in CAT_FILL_UNKNOWN:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna('Unknown')

df_clean['priority'] = df_clean['priority'].fillna(df_clean['priority'].mode()[0])
df_clean['requires_road_closure'] = df_clean['requires_road_closure'].astype(int)

In [8]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [9]:
LE_COLS = ['event_type', 'event_cause', 'veh_type', 'status']
le_encoders = {}
for col in LE_COLS:
    if col in df_clean.columns:
        le = LabelEncoder()
        df_clean[col + '_enc'] = le.fit_transform(df_clean[col].astype(str))
        le_encoders[col] = le

# High-cardinality: target encoding for zone, junction
# (avoids OHE explosion; robust for tree models)
target_col = 'duration_mins'
for col in ['zone', 'junction', 'corridor', 'gba_identifier']:
    if col in df_clean.columns:
        mean_map = df_clean.groupby(col)[target_col].mean()
        df_clean[col + '_tenc'] = df_clean[col].map(mean_map)
        # Add binary "is_known" flag alongside
        df_clean[col + '_known'] = (df_clean[col] != 'Unknown').astype(int)

# Priority ordinal encoding
priority_map = {'low': 0, 'medium': 1, 'high': 2, 'critical': 3}
if 'priority' in df_clean.columns:
    df_clean['priority_enc'] = df_clean['priority'].str.lower().map(priority_map)

In [10]:
FEATURE_COLS = [
    # temporal
    'hour', 'day_of_week', 'month', 'is_weekend', 'is_peak', 'is_night',
    # geo
    'latitude', 'longitude', 'lat_bin', 'lon_bin', 'has_end_location',
    # event attributes
    'event_type_enc', 'event_cause_enc', 'requires_road_closure',
    'veh_type_enc', 'status_enc',
    # encoded high-card
    'zone_tenc', 'zone_known',
    'junction_tenc', 'junction_known',
    'corridor_tenc', 'corridor_known',
    'gba_identifier_tenc', 'gba_identifier_known',
    # priority (when not the target)
    'priority_enc',
]
# Keep only columns that exist after all the steps above
FEATURE_COLS = [c for c in FEATURE_COLS if c in df_clean.columns]

In [11]:
print(f"Final feature count: {len(FEATURE_COLS)}")
print(f"Final dataset size:  {len(df_clean)} rows")

Final feature count: 25
Final dataset size:  8173 rows


In [12]:
# training

In [13]:
from sklearn.model_selection import cross_val_score, StratifiedKFold, KFold
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    f1_score, roc_auc_score, classification_report, confusion_matrix
)
import xgboost as xgb
import lightgbm as lgb

In [14]:
X = df_clean[FEATURE_COLS].fillna(-999)   # -999 sentinel for trees
y_reg   = df_clean['duration_mins']
y_clf_b = df_clean['requires_road_closure']   # binary
y_clf_m = df_clean['severity'].cat.codes       # multi-class (0=low,1=med,2=high,3=critical)

X_train, X_test, y_reg_train, y_reg_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)
_, _, y_b_train, y_b_test = train_test_split(X, y_clf_b, test_size=0.2, random_state=42)
_, _, y_m_train, y_m_test = train_test_split(X, y_clf_m, test_size=0.2, random_state=42)

def regression_report(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f"\n{'─'*40}")
    print(f"  {name}")
    print(f"  MAE : {mae:.2f} mins")
    print(f"  RMSE: {rmse:.2f} mins")
    print(f"  R²  : {r2:.4f}")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2}

def classification_report_short(name, y_true, y_pred, y_prob=None):
    f1 = f1_score(y_true, y_pred, average='weighted')
    auc = roc_auc_score(y_true, y_prob, multi_class='ovr', average='weighted') \
          if y_prob is not None and y_prob.ndim == 2 else None
    print(f"\n{'─'*40}")
    print(f"  {name}")
    print(f"  F1 (weighted): {f1:.4f}")
    if auc: print(f"  ROC-AUC:       {auc:.4f}")
    return {'model': name, 'F1': f1, 'AUC': auc}

In [16]:
df['closed_datetime']  = pd.to_datetime(df['closed_datetime'],  utc=True, errors='coerce')
df['start_datetime']   = pd.to_datetime(df['start_datetime'],   utc=True, errors='coerce')
df['resolved_datetime'] = pd.to_datetime(df['resolved_datetime'], utc=True, errors='coerce')

# Step 2: best available end time
df['end_ts'] = df['closed_datetime'].fillna(df['resolved_datetime'])

# Step 3: compute duration
df['duration_mins'] = (df['end_ts'] - df['start_datetime']).dt.total_seconds() / 60

# Step 4: filter — strict, no NaN leaking through
mask = (
    df['duration_mins'].notna() &
    (df['duration_mins'] > 0) &
    (df['duration_mins'] < 1440)
)
df_clean = df[mask].copy()
print(f"Usable rows after duration filter: {len(df_clean)}")

# Verify — must be 0
print(f"NaN in duration_mins: {df_clean['duration_mins'].isna().sum()}")

Usable rows after duration filter: 2523
NaN in duration_mins: 0


In [17]:
print("\n" + "="*50)
print("  TASK A: DURATION REGRESSION")
print("="*50)

results_reg = []

# ── Baseline: Ridge regression ────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

ridge_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0))
])
ridge_pipe.fit(X_train, y_reg_train)
pred = ridge_pipe.predict(X_test)
results_reg.append(regression_report("Ridge (baseline)", y_reg_test, pred))

# ── Random Forest ─────────────────────────────────────────────────────────────
rf_reg = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf_reg.fit(X_train, y_reg_train)
pred = rf_reg.predict(X_test)
results_reg.append(regression_report("RandomForest", y_reg_test, pred))

# ── XGBoost ───────────────────────────────────────────────────────────────────
xgb_reg = xgb.XGBRegressor(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, eval_metric='mae',
    early_stopping_rounds=30, verbosity=0
)
xgb_reg.fit(
    X_train, y_reg_train,
    eval_set=[(X_test, y_reg_test)],
    verbose=False
)
pred = xgb_reg.predict(X_test)
results_reg.append(regression_report("XGBoost", y_reg_test, pred))

# ── LightGBM ──────────────────────────────────────────────────────────────────
lgb_reg = lgb.LGBMRegressor(
    n_estimators=500, max_depth=7, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbose=-1
)
lgb_reg.fit(
    X_train, y_reg_train,
    eval_set=[(X_test, y_reg_test)],
    callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(period=-1)]
)
pred = lgb_reg.predict(X_test)
results_reg.append(regression_report("LightGBM", y_reg_test, pred))

# ── Summary table ─────────────────────────────────────────────────────────────
import pandas as pd
print("\n\nTask A Summary:")
print(pd.DataFrame(results_reg).sort_values('MAE').to_string(index=False))


  TASK A: DURATION REGRESSION


ValueError: Input y contains NaN.